# 🎵 ACAIAMUSIC · Estúdio de música (GPU gratuita do Kaggle)

Este notebook liga **o seu próprio estúdio de IA** (ACE-Step 1.5, modelo aberto) na GPU gratuita do Kaggle.

**Antes de rodar (só na primeira vez):**
1. No painel da direita → **Session options** → **Accelerator**: escolha **GPU T4 x2** (ou GPU P100).
2. Em **Internet**: deixe **ligado** (exige conta com telefone verificado).

**Para usar:** clique em **Run All** (▶▶). Na primeira vez leva uns 10–15 minutos (baixa o modelo). No final aparece um quadro com o **endereço** e a **chave**: copie os dois para o app ACAIAMUSIC em ⚙️ *Servidor de música*.

⚠️ Deixe esta aba aberta enquanto estiver criando músicas. O Kaggle dá **30 horas de GPU por semana** grátis e cada sessão dura até 12 horas. Quando terminar, clique em **Stop session** para economizar horas.

In [ ]:
# 1) Instala o ACE-Step 1.5 (só demora na primeira vez da sessão)
import os, subprocess
os.chdir('/kaggle/working')
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f'Falhou: {cmd}')

os.environ['UV_LINK_MODE'] = 'copy'
# Confere se a GPU está ligada (sem GPU o estúdio não funciona)
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit('❌ A GPU está DESLIGADA. Vá em Settings → Accelerator → GPU T4 x2, confirme e clique em Run All de novo.')
print('✅ GPU encontrada:', gpu.stdout.strip())
if not os.path.isdir('ACE-Step-1.5'):
    sh('git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git')
sh('curl -LsSf https://astral.sh/uv/install.sh | sh')
sh('cd ACE-Step-1.5 && uv sync')
print('\n✅ Instalação concluída')

In [ ]:
# 2) Liga o servidor de música e cria o endereço público
import os, re, time, secrets, threading, subprocess, urllib.request
from IPython.display import display, HTML

KEY = secrets.token_urlsafe(18)
env = dict(os.environ,
           ACESTEP_API_HOST='127.0.0.1', ACESTEP_API_PORT='8001', ACESTEP_API_KEY=KEY,
           ACESTEP_CONFIG_PATH='acestep-v15-turbo',
           ACESTEP_INIT_LLM='true', ACESTEP_LM_MODEL_PATH='acestep-5Hz-lm-1.7B', ACESTEP_LM_BACKEND='pt',
           MPLBACKEND='Agg')  # o Kaggle define um backend de gráficos que não existe no estúdio
log = open('/kaggle/working/acestep.log', 'w')
server = subprocess.Popen(['uv', 'run', 'acestep-api'], cwd='/kaggle/working/ACE-Step-1.5',
                          env=env, stdout=log, stderr=subprocess.STDOUT)

def mostrar_erro():
    linhas = open('/kaggle/working/acestep.log', errors='ignore').read().splitlines()
    chaves = ('Error', 'error', 'Traceback', 'CUDA', 'No module', 'Exception', 'Killed', 'memory')
    importantes = [l for l in linhas if any(p in l for p in chaves)]
    print('\n'.join((importantes or linhas)[-40:]))

print('⏳ Carregando o modelo de música (pode levar alguns minutos)…')
ready = False
for i in range(360):
    if server.poll() is not None:
        mostrar_erro()
        raise SystemExit('❌ O servidor parou. Mande um print das linhas acima.')
    try:
        req = urllib.request.Request('http://127.0.0.1:8001/v1/stats', headers={'Authorization': 'Bearer ' + KEY})
        urllib.request.urlopen(req, timeout=5)
        ready = True
        break
    except Exception:
        time.sleep(5)
        if i % 12 == 11:  # a cada ~1 minuto mostra que está vivo
            ult = [l for l in open('/kaggle/working/acestep.log', errors='ignore').read().splitlines() if l.strip()]
            print(f'   … {(i + 1) // 12} min · ainda carregando · {ult[-1][:120] if ult else "baixando o modelo"}')
if not ready:
    mostrar_erro()
    raise SystemExit('❌ O servidor não respondeu a tempo.')
print('✅ Servidor de música ligado')

# Túnel gratuito da Cloudflare (sem conta) para o app acessar o servidor
import queue
print('🌐 Criando o endereço público (túnel da Cloudflare)…')
CF = '/kaggle/working/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)
tunnel = subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8001'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
linhas_tunel = queue.Queue()
threading.Thread(target=lambda: [linhas_tunel.put(l) for l in tunnel.stdout], daemon=True).start()
URL, fim, visto = None, time.time() + 120, []
while time.time() < fim and URL is None:
    try:
        line = linhas_tunel.get(timeout=5)
    except queue.Empty:
        continue
    visto.append(line.strip())
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        URL = m.group(0)
if URL is None:
    print('\n'.join(visto[-15:]))
    raise SystemExit('❌ Não consegui criar o endereço público. Rode esta célula (2) de novo; se repetir, mande um print.')
print('✅ Endereço criado')

display(HTML(f'''
<div style="font-family:sans-serif;padding:22px;border-radius:16px;background:#14141b;color:#fff;border:2px solid #ec4899">
  <h2 style="margin:0 0 12px">🎵 Seu estúdio ACAIAMUSIC está no ar!</h2>
  <p style="margin:4px 0">Copie para o app em <b>⚙️ Servidor de música</b>:</p>
  <p style="margin:10px 0 4px;color:#f9a8d4">ENDEREÇO</p>
  <code style="font-size:18px;background:#000;padding:8px 12px;border-radius:8px;display:inline-block">{URL}</code>
  <p style="margin:14px 0 4px;color:#f9a8d4">CHAVE</p>
  <code style="font-size:18px;background:#000;padding:8px 12px;border-radius:8px;display:inline-block">{KEY}</code>
  <p style="margin:16px 0 0;color:#aaa">Deixe esta aba aberta enquanto usa. O endereço muda toda vez que você reinicia o notebook.</p>
</div>'''))

In [ ]:
# 3) Mantém o estúdio ligado e mostra o que está acontecendo (pare a sessão quando terminar)
import time
start = time.time()
while server.poll() is None:
    mins = int((time.time() - start) / 60)
    tail = open('/kaggle/working/acestep.log').read()[-300:].strip().splitlines()
    print(f'[{mins} min] estúdio ligado · {URL} · último log: {tail[-1] if tail else "-"}'[:300])
    time.sleep(60)
print('O servidor parou. Rode a célula 2 de novo.')